# B16 Macro Scenario Generator

In [ ]:
!pip install --upgrade langgraph langchain langchain-community langchain-openai openai yfinance PyPortfolioOpt cvxpy ecos scs osqp clarabel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict, Any, Optional, Tuple
import json
import time
import re
from __future__ import annotations
import math, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import cvxpy as cp
from pypfopt import risk_models, expected_returns
import requests

warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True)
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [ ]:
# Create the LLM
llm = ChatOpenAI(
    model="openai/gpt-oss-20b:free",  # replace with the model you want to use from OpenRouter
    openai_api_key=os.environ["OPENAI_API_KEY"],
    openai_api_base=os.environ["OPENAI_BASE_URL"],
    temperature=0.2
)
# Test it
response = llm.invoke("Give me a 1-sentence macroeconomic summary of a Fed rate hike.")
print(response.content)

A Fed rate hike signals tighter monetary policy aimed at curbing inflation, which typically raises borrowing costs, dampens consumer spending and investment, and can slow economic growth while potentially stabilizing price levels.


Simulate a Fed rate hike macro scenario.

In [ ]:
answer = input("Please enter a macroeconomic scneario you want to simulate.")

Please enter a macroeconomic scneario you want to simulate.Simulate a Fed rate hike macro scenario.


In [ ]:
# Simple example of a scenario generation node
def scenario_node(state):
    scenario = llm.invoke(answer)
    return {"scenario": scenario.content}

# Define graph
graph = StateGraph(dict)
graph.add_node("scenario", scenario_node)
graph.set_entry_point("scenario")
graph.set_finish_point("scenario")

# Run it
app = graph.compile()
scenario_text = app.invoke({})
print(scenario_text)


{'scenario': 'Below is a **“what‑if” macro‑simulation** that walks through a single Fed rate‑hike cycle, from the policy decision to the ripple effects across the economy, markets, and the balance sheet.  The numbers are illustrative (not a forecast) and are meant to show the typical chain of reactions that economists and central‑bank analysts expect when the Fed raises its policy rate.\n\n---\n\n## 1. Baseline (Pre‑Hike)\n\n| Variable | Value | Notes |\n|----------|-------|-------|\n| **Fed Funds Target** | 4.75\u202f% | 25\u202fbp above the 4.50\u202f% “neutral” level |\n| **Inflation (CPI, YoY)** | 4.2\u202f% | Above the 2\u202f% target |\n| **GDP Growth (annualized)** | 2.5\u202f% | Moderately strong |\n| **Unemployment** | 4.0\u202f% | Near full‑employment |\n| **10‑yr Treasury Yield** | 3.10\u202f% | Reflects market expectations |\n| **S&P\u202f500** | 4,200 | 12‑month high |\n| **USD Index** | 94 | Slightly above 2023 average |\n| **Housing Starts** | 1.2\u202fM | 3\u202f% YoY g

In [ ]:
class ScenarioState(TypedDict):
    scenario_text: str
    macro_analysis: Dict[str, Any]
    critique: str
    revision: Dict[str, Any]
    evaluation: str
def clean_json(text):
    # Replace fancy quotes and dashes with standard ones
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("’", "'").replace("–", "-").replace("-", "-").replace(" ", " ")
    # Extract the JSON part only
    match = re.search(r"\{.*\}", text, re.S)
    if match:
        text = match.group()
    # Remove trailing commas
    text = re.sub(r",(\s*[\}\]])", r"\1", text)
    return text.strip()

# -----------------------------
# Agent 1: LLM extraction
# -----------------------------
def llm_extraction_agent(state: ScenarioState) -> ScenarioState:
    text = state["scenario_text"]

    prompt = f"""
    You are an analyst. Read the scenario below and:
    1. Identify macro factor changes, with direction and percentage (1%, -2%). Even if the baseline value is not provided, please predict the effect of all the factors listed in the macro factor list.
    The macro factor list is: [
    'CPIAUCSL', 'CPILFESL', 'PCEPI', 'GDPDEF',  # Inflation
    'UNRATE', 'PAYEMS', 'CIVPART', 'ICSA',      # Employment
    'FEDFUNDS', 'DGS10', 'TB3MS', 'MORTGAGE30US', 'T10Y2Y', 'BAA', 'BAA10Y', # Interest Rates
    'MSPNHSUS', 'HOUST', 'CSUSHPINSA', 'PERMIT', # Housing
    'GDP', 'GDPC1', 'INDPRO', 'RETAILSMNSA',     # GDP and Output
    'UMCSENT', 'BUSINV',                         # Sentiment and Confidence
    'VIXCLS', 'INDPRO','M2SL'
]

    Return ONLY a valid JSON object in the following format:

{{

    "CPIAUCSL": {{"change": "-3%", "reason": "The Fed’s 25 bp hike in June 2025 immediately raised borrowing costs and cooled inflation."}},
    "UNRATE": {{"change": "+0.5%", "reason": "Higher interest rates slowed hiring, pushing unemployment slightly higher."}},
    ...

}}

Rules:
- The JSON must be syntactically valid (no markdown, no comments, no extra text).
- Include all factors from the list above (even if the effect is neutral, set "change": "0%" and explain briefly why).
- Do not include any explanation outside of the JSON.
- Do not include any other text.

Scenario:
    {text}
    """
    response = llm.invoke(prompt)
    print("--- Raw LLM Response ---")
    print(response.content)
    parsed = clean_json(response.content)
    print("--- Cleaned LLM Response ---")
    print(parsed)
    result = json.loads(parsed)
    state["macro_analysis"] = result
    return state

# # -----------------------------
# # Build LangGraph
# # -----------------------------
# def build_graph():
#     g = StateGraph(ScenarioState)
#     g.add_node("llm_extraction_agent", llm_extraction_agent)
#     g.set_entry_point("llm_extraction_agent")
#     g.add_edge("llm_extraction_agent", END)
#     return g.compile()

# graph = build_graph()
# final_state = graph.invoke({"scenario_text": scenario_text})

In [ ]:
# -----------------------------
# Agent 2: Critic
# -----------------------------
def critic_agent(state: ScenarioState) -> ScenarioState:
    analysis_json = json.dumps(state["macro_analysis"], indent=2)

    prompt = f"""
    You are a senior economist reviewing a macroeconomic analysis.

    Review the following JSON for:
    1. Missing or implausible factors.
    2. Logical inconsistencies between related indicators (e.g., inflation vs. interest rates).
    3. Poor reasoning or vague explanations.

    Respond with plain text feedback — no JSON, no markdown.

    Analysis:
    {analysis_json}
    """

    response = llm.invoke(prompt)
    state["critique"] = response.content
    # print(json.dumps(state, indent=2))

    return state

In [ ]:
# -----------------------------
# Agent 3: Revision
# -----------------------------
def revision_agent(state: ScenarioState) -> ScenarioState:
    critique = json.dumps(state["critique"], indent=2)
    original_json = json.dumps(state["macro_analysis"], indent=2)

    prompt = f"""
    You are a macroeconomist revising an analysis.

    Using the critique below, produce a corrected and improved version of the JSON.
    Keep the same format and include all macro factors.

    Critique:
    {critique}

    Original JSON:
    {original_json}

    Return ONLY the corrected JSON.
    """

    response = llm.invoke(prompt)
    parsed = clean_json(response.content)
    revised = json.loads(parsed)
    state["revision"] = revised
    print(json.dumps(state, indent=2))
    return state


In [ ]:
# -----------------------------
# Agent 4: Evaluation
# -----------------------------
def evaluation_agent(state: ScenarioState) -> ScenarioState:
    revised_json = json.dumps(state["revision"], indent=2)

    prompt = f"""
    Evaluate the revised macroeconomic analysis below.
    Score it (1–10) for:
    - Completeness (covers all factors)
    - Economic logic
    - Clarity of reasoning

    Return your evaluation in plain text.
    Revised JSON:
    {revised_json}
    """

    response = llm.invoke(prompt)
    state["evaluation"] = response.content.strip()
    return state

In [ ]:
# -----------------------------
# Graph Build
# -----------------------------
def build_graph():
    g = StateGraph(ScenarioState)

    g.add_node("llm_extraction_agent", llm_extraction_agent)
    g.add_node("critic_agent", critic_agent)
    g.add_node("revision_agent", revision_agent)
    g.add_node("evaluation_agent", evaluation_agent)

    # Define a conditional edge function
    def should_continue(state):
        # Extract the score from the evaluation.
        # This assumes the evaluation agent's output contains a score.
        # You might need to adjust this based on the actual output format.
        evaluation_text = state.get("evaluation", "")
        # Simple check for a score in the evaluation text (e.g., "Score: 8/10")
        match = re.search(r"(\d+)/10", evaluation_text)
        if match:
            score = int(match.group(1))
            # Set a threshold for needing further revision
            if score < 7:  # Example threshold: revise if score is less than 7
                print(f"Evaluation score {score}/10 is below threshold. Revising...")
                return "continue"  # Return the key for the next node in the mapping
            else:
                print(f"Evaluation score {score}/10 is sufficient. Ending.")
                return "end" # Return the key for ending the graph
        else:
            print("Could not parse evaluation score. Ending.")
            return "end" # Return the key for ending the graph


    # Flow: Extraction → Critic → Revision → Evaluation
    g.add_edge("llm_extraction_agent", "critic_agent")
    g.add_edge("critic_agent", "revision_agent")
    g.add_edge("revision_agent", "evaluation_agent")

    # Add conditional edge from evaluation back to critic or end
    g.add_conditional_edges("evaluation_agent", should_continue, {
        "continue": "critic_agent",
        "end": END
    })


    g.set_entry_point("llm_extraction_agent")
    return g.compile()
graph = build_graph()
final_state = graph.invoke({"scenario_text": scenario_text})
print("\n=== FINAL STATE ===")
print(json.dumps(final_state, indent=2))

--- Raw LLM Response ---
{
    "CPIAUCSL": {"change": "-2.3%", "reason": "Fed hikes raised borrowing costs, cooling demand and reducing CPI from 4.2% to 1.9% by Year 2."},
    "CPILFESL": {"change": "-1.5%", "reason": "Core CPI followed the headline trend, falling as inflationary pressures eased."},
    "PCEPI": {"change": "-1.8%", "reason": "Personal Consumption Expenditures price index mirrored CPI declines due to tighter credit."},
    "GDPDEF": {"change": "-1.5%", "reason": "GDP deflator fell as overall price growth slowed with higher rates."},
    "UNRATE": {"change": "+0.7%", "reason": "Higher rates slowed hiring, pushing unemployment from 4.0% to 4.7%."},
    "PAYEMS": {"change": "-0.5%", "reason": "Employment numbers fell in line with the modest rise in unemployment."},
    "CIVPART": {"change": "0%", "reason": "Labor force participation remained largely unchanged during the cycle."},
    "ICSA": {"change": "0%", "reason": "Industrial production index showed no net change beyon

In [ ]:
# Define the path to save the file in Google Drive
save_dir = '/content/drive/MyDrive/4511 B16/'
save_path = os.path.join(save_dir, 'final_state.json')
with open(save_path, 'w') as f:
    json.dump(final_state, f, indent=2)

print(f"Final state saved to: {save_path}")

Final state saved to: /content/drive/MyDrive/4511 B16/final_state.json


# B16 Portfolio Analyzer + Risk Simulator

In [ ]:
PRICE_CSV = '/content/drive/MyDrive/4511 B16/sp500_daily_data.csv'
FRED_CSV  = '/content/drive/MyDrive/4511 B16/fred_data_quarterly.csv'

OUTPUT_DIR = Path("/content/drive/MyDrive/4511 B16/Stage2_Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


RNG = np.random.default_rng(42)

In [ ]:
# Loading Data
def load_sp500_prices(csv_path: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(csv_path, header=[0,1], index_col=0)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    # Expect columns like ('Close','AAPL'), ('Volume','AAPL'), etc.
    df = df.sort_index(axis=1)

    # Close & Volume
    close = df['Close'].copy()
    volume = df['Volume'].copy() if ('Volume' in df.columns.get_level_values(0)) else None

    # Coerce numeric
    close = close.apply(pd.to_numeric, errors='coerce')
    if volume is not None:
        volume = volume.apply(pd.to_numeric, errors='coerce')

    # Returns
    ret = close.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return close, volume, ret

def load_fred_quarterly(csv_path: str) -> pd.DataFrame:
    fred = pd.read_csv(csv_path)
    # Flexible date column handling: first col assumed to be date
    date_col = fred.columns[0]
    fred[date_col] = pd.to_datetime(fred[date_col])
    fred = fred.set_index(date_col).sort_index()

    # Compute YoY % change for each indicator
    fred_yoy = fred.pct_change(4)
    fred_yoy = fred_yoy.replace([np.inf, -np.inf], np.nan)

    return fred_yoy

close_wide, volume_wide, ret_wide = load_sp500_prices(PRICE_CSV)
fred_yoy = load_fred_quarterly(FRED_CSV)
print(close_wide.shape, fred_yoy.shape)

(6289, 841) (101, 31)


In [ ]:
# Daily factor set

# Market factor = equal-weighted return across all available tickers
mkt = ret_wide.mean(axis=1).to_frame('MKT')

# PCA factors from cross-section to capture common drivers
K = 3
valid = ret_wide.dropna(axis=1, how='all').fillna(0.0)
pca = PCA(n_components=K, random_state=42)
pca_scores = pca.fit_transform(valid.values)
pca_df = pd.DataFrame(pca_scores, index=valid.index, columns=[f'PC{i+1}' for i in range(K)])

# Expand FRED QoQ changes to daily by forward fill per quarter
fred_daily = fred_yoy.reindex(mkt.index, method='ffill').fillna(method='ffill')

factors = pd.concat([mkt, pca_df, fred_daily], axis=1)
factors = factors.replace([np.inf, -np.inf], np.nan).fillna(0.0)
factors.to_csv(OUTPUT_DIR / 'factors_daily.csv')
print('Factors shape:', factors.shape)

Factors shape: (6289, 35)


In [ ]:
# Input Router Agent (Not sure if Qiyong covered this)
ROUTER_EVENT_ONLY   = 'EVENT_ONLY'
ROUTER_HOLDINGS_ONLY= 'HOLDINGS_ONLY'
ROUTER_BOTH         = 'BOTH'

def make_default_portfolio(n_tickers: int = 50) -> pd.DataFrame:

    cols = list(close_wide.columns)
    if volume_wide is not None:
        med_vol = volume_wide.median(axis=0).sort_values(ascending=False)
        tickers = list(med_vol.head(n_tickers).index)
    else:
        tickers = cols[:n_tickers]
    w = np.repeat(1.0/len(tickers), len(tickers))
    return pd.DataFrame({'Ticker': tickers, 'Weight': w})

def route_input(natural_text: Optional[str], holdings_df: Optional[pd.DataFrame], scenario_json: Optional[Dict[str,Any]]):
    # Determine mode
    has_event = scenario_json is not None and len(scenario_json)>0
    has_hold  = holdings_df is not None and not holdings_df.empty
    if has_event and has_hold:
        mode = ROUTER_BOTH
    elif has_event:
        mode = ROUTER_EVENT_ONLY
    elif has_hold:
        mode = ROUTER_HOLDINGS_ONLY
    else:
        mode = ROUTER_EVENT_ONLY  # default to run a scenario on default buckets

    # Portfolio
    if not has_hold:
        portfolio_df = make_default_portfolio(50)
    else:
        portfolio_df = holdings_df.copy()
        if 'Weight' not in portfolio_df.columns:
            portfolio_df['Weight'] = 1.0/len(portfolio_df)
        # Normalize
        portfolio_df['Weight'] = portfolio_df['Weight'].astype(float)
        s = portfolio_df['Weight'].sum()
        portfolio_df['Weight'] = portfolio_df['Weight'] / (s if s!=0 else 1.0)

    # Scenario
    if not has_event:
        # Baseline 'neutral' scenario — zero changes
        scenario_json = {}
    return mode, portfolio_df, scenario_json

print('Router ready.')

Router ready.


In [ ]:
# Portfolio Analyzer
def fit_exposures(portfolio: pd.DataFrame, factors: pd.DataFrame, ret_wide: pd.DataFrame, alpha: float = 1.0):
    tickers = [t for t in portfolio['Ticker'].tolist() if t in ret_wide.columns]
    if len(tickers)==0:
        raise ValueError('No overlapping tickers between portfolio and price data.')
    # Align
    R = ret_wide[tickers].loc[factors.index].fillna(0.0)
    X = factors.values
    Fnames = list(factors.columns)
    betas = []
    resid_vars = []
    for i, t in enumerate(tickers):
        y = R[t].values
        mdl = Ridge(alpha=alpha, fit_intercept=True)
        mdl.fit(X, y)
        y_hat = mdl.predict(X)
        resid = y - y_hat
        resid_var = np.var(resid, ddof=1)
        resid_vars.append(resid_var)
        beta_row = dict(Ticker=t)
        beta_row.update({f: b for f,b in zip(Fnames, mdl.coef_)})
        beta_row['Intercept'] = mdl.intercept_
        betas.append(beta_row)
    betas_df = pd.DataFrame(betas).set_index('Ticker')
    resid_var_s = pd.Series(resid_vars, index=tickers, name='Sigma2_e')
    # Portfolio factor exposure = sum_i w_i * beta_i
    w = portfolio.set_index('Ticker')['Weight'].reindex(betas_df.index).fillna(0.0)
    port_expo = (betas_df.mul(w, axis=0)).sum(axis=0)
    # Risk decomposition using factor cov
    Sigma_f = np.cov(factors.values.T)
    # portfolio factor variance
    port_var_factor = float(port_expo[Fnames].values @ Sigma_f @ port_expo[Fnames].values.T)
    # idiosyncratic variance
    port_var_idio = float((w.pow(2) * resid_var_s).sum())
    port_var = port_var_factor + port_var_idio
    contrib_to_var = pd.Series(np.diag(np.outer(port_expo[Fnames].values, port_expo[Fnames].values) @ Sigma_f), index=Fnames, name='VarContrib')
    out = {
        'betas_df': betas_df,
        'resid_var': resid_var_s,
        'portfolio_exposure': port_expo,
        'Sigma_f': Sigma_f,
        'factor_names': Fnames,
        'portfolio_var_factor': port_var_factor,
        'portfolio_var_idio': port_var_idio,
        'portfolio_var_total': port_var,
        'contrib_to_var': contrib_to_var
    }
    return out

print('Analyzer ready.')

Analyzer ready.


In [ ]:
# Shock mapping from macro JSON
BP = 0.0001
def parse_change_to_decimal(s: str) -> float:
    s = str(s).strip().lower()

    # basis points / bp
    if "basis point" in s:
        m = re.search(r'([+\-]?\s*\d+(\.\d+)?)', s)
        return float(m.group(1)) * 0.0001

    if "bp" in s:
        m = re.search(r'([+\-]?\s*\d+(\.\d+)?)', s)
        return float(m.group(1)) * 0.0001

    # percent %
    if "%" in s:
        m = re.search(r'([+\-]?\s*\d+(\.\d+)?)', s)
        return float(m.group(1)) / 100.0

    # numeric fallback
    try:
        return float(s)
    except:
        return 0.0

def collect_shocks_recursive(node, shocks):
    if isinstance(node, dict):
        if "change" in node:
            shocks[node.get("name", None)] = node["change"]

        for k, v in node.items():
            if isinstance(v, dict) and "change" in v:
                shocks[k] = v["change"]
            else:
                collect_shocks_recursive(v, shocks)

    elif isinstance(node, list):
        for item in node:
            collect_shocks_recursive(item, shocks)

def macro_to_factor_shift(scenario_json: Dict[str, Any], factor_names: list[str]) -> np.ndarray:
    shocks = {}
    collect_shocks_recursive(scenario_json, shocks)

    mu = np.zeros(len(factor_names), dtype=float)

    # factor index
    idx = {f: i for i, f in enumerate(factor_names)}

    def add(factor, val):
        if factor in idx:
            mu[idx[factor]] += val

    GROWTH_KEYS = {"GDP", "GDPC1", "INDPRO", "PAYEMS", "RETAILSMNSA"}
    INFLATION_KEYS = {"CPIAUCSL", "CPILFESL", "PCEPI", "COREPCE", "GDPDEF"}
    UNEMP_KEYS = {"UNRATE", "CIVPART", "ICSA"}
    FED_KEYS = {"FEDFUNDS", "TB3MS", "DGS10"}
    YIELDCURVE_KEYS = {"T10Y2Y"}
    CREDIT_KEYS = {"BAA", "BAA10Y"}
    VOL_KEYS = {"VIXCLS"}
    HOUSING_KEYS = {"MSPNHSUS", "HOUST", "CSUSHPINSA", "PERMIT"}
    MONEY_KEYS = {"M2SL"}

    for key, change_str in shocks.items():
        key_up = key.upper()
        delta = parse_change_to_decimal(change_str)

        # ---- Inflation -> PC2 ----
        if key_up in INFLATION_KEYS:
            add("MKT", -0.3 * delta)
            add("PC1", -0.1 * delta)
            add("PC2", +1.0 * delta)

        # ---- Growth -> PC1 ----
        elif key_up in GROWTH_KEYS:
            add("MKT", +0.2 * delta)
            add("PC1", +1.0 * delta)
            add("PC2", -0.1 * delta)

        # ---- Labor market -> PC1 + PC3 ----
        elif key_up in UNEMP_KEYS:
            add("MKT", -0.3 * delta)
            add("PC1", -1.0 * delta)
            add("PC3", +0.1 * delta)

        # ---- Fed policy / rates ----
        elif key_up in FED_KEYS:
            add("MKT", -0.5 * delta)
            add("PC1", -0.3 * delta)
            add("PC2", -0.2 * delta)
            add("PC3", +0.4 * delta)

        # ---- Yield curve ----
        elif key_up in YIELDCURVE_KEYS:
            add("PC1", +0.4 * delta)
            add("PC3", +1.0 * delta)

        # ---- Credit spreads ----
        elif key_up in CREDIT_KEYS:
            add("MKT", -0.5 * delta)
            add("PC1", -0.4 * delta)
            add("PC3", +1.0 * delta)

        # ---- Volatility ----
        elif key_up in VOL_KEYS:
            add("MKT", -0.8 * delta)
            add("PC3", +0.8 * delta)

        # ---- Housing ----
        elif key_up in HOUSING_KEYS:
            add("MKT", +0.2 * delta)
            add("PC1", +0.8 * delta)
            add("PC3", -0.2 * delta)

        # ---- Money Supply ----
        elif key_up in MONEY_KEYS:
            add("MKT", +0.2 * delta)
            add("PC1", +0.3 * delta)
            add("PC2", +0.3 * delta)

        else:
            add("MKT", 0.1 * delta)

    return mu

print('Shock mapper ready.')

Shock mapper ready.


In [ ]:
# Monte Carlo Risk Simulator
def mc_portfolio_pl(port_expo: pd.Series, Sigma_f: np.ndarray, factor_names: list[str], mean_shift: np.ndarray,
                    resid_var: pd.Series, weights: pd.Series, n_paths: int = 20000, seed: int = 42):
    rng = np.random.default_rng(seed)

    # Draw factor returns
    F = rng.multivariate_normal(mean=mean_shift, cov=Sigma_f, size=n_paths)

    # Factor P&L for portfolio
    port_factor_expo = port_expo[factor_names].values  # vector length = K
    pnl_factor = F @ port_factor_expo

    # Idiosyncratic P&L ~ N(0, sum w_i^2 * sigma2_ei)
    sigma2_idio = float((weights.pow(2) * resid_var.reindex(weights.index).fillna(0.0)).sum())
    pnl_idio = rng.normal(loc=0.0, scale=math.sqrt(max(sigma2_idio,0.0)), size=n_paths)
    pnl = pnl_factor + pnl_idio
    return pnl

def var_cvar(pnl: np.ndarray, alpha: float = 0.95) -> Tuple[float, float]:
    # Loss = -PnL convention
    loss = -pnl
    var = np.quantile(loss, alpha)
    cvar = loss[loss>=var].mean() if np.any(loss>=var) else var
    return float(var), float(cvar)

def plot_pnl(pnl: np.ndarray, out_path: Path):
    plt.figure()
    plt.hist(pnl, bins=80)
    plt.title('Scenario P&L distribution (MC)')
    plt.xlabel('P&L (fractional return)')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

print('Monte Carlo Simulator ready.')

Monte Carlo Simulator ready.


In [ ]:
# Pipeline runner
def run_pipeline(natural_text: Optional[str] = None,
                 holdings_df: Optional[pd.DataFrame] = None,
                 scenario_json: Optional[Dict[str,Any]] = None,
                 alpha_ridge: float = 1.0,
                 n_paths: int = 20000,
                 seed: int = 42):
    mode, portfolio_df, scenario_json = route_input(natural_text, holdings_df, scenario_json)
    print(f"Mode: {mode}; Portfolio size: {len(portfolio_df)}; Scenario keys: {list(scenario_json.keys()) if scenario_json else []}")

    # 1) Exposures
    ana = fit_exposures(portfolio_df, factors, ret_wide, alpha=alpha_ridge)
    betas_df = ana['betas_df']
    betas_df.to_csv(OUTPUT_DIR / 'exposures_table.csv')
    contrib = ana['contrib_to_var'].sort_values(ascending=False)
    contrib.to_csv(OUTPUT_DIR / 'risk_contrib.csv')
    print('Portfolio variance (factor, idio, total)=', ana['portfolio_var_factor'], ana['portfolio_var_idio'], ana['portfolio_var_total'])

    # 2) Shock mapping
    mu_shift = macro_to_factor_shift(scenario_json, ana['factor_names'])
    np.savetxt(OUTPUT_DIR / 'factor_mean_shift.txt', mu_shift[None,:])

    # 3) Monte Carlo
    w = portfolio_df.set_index('Ticker')['Weight']
    pnl = mc_portfolio_pl(ana['portfolio_exposure'], ana['Sigma_f'], ana['factor_names'], mu_shift, ana['resid_var'], w, n_paths=n_paths, seed=seed)
    np.savetxt(OUTPUT_DIR / 'pnl_paths.txt', pnl)
    v, cv = var_cvar(pnl, alpha=0.95)
    print(f"VaR95={v:.4f}, CVaR95={cv:.4f}")
    plot_pnl(pnl, OUTPUT_DIR / 'pnl_hist.png')

    # 4) Summary JSON
    summary = {
        'mode': mode,
        'portfolio_size': int(len(portfolio_df)),
        'factor_names': ana['factor_names'],
        'portfolio_var_factor': float(ana['portfolio_var_factor']),
        'portfolio_var_idio': float(ana['portfolio_var_idio']),
        'portfolio_var_total': float(ana['portfolio_var_total']),
        'VaR95': float(v),
        'CVaR95': float(cv)
    }
    with open(OUTPUT_DIR / 'summary.json','w') as f:
        json.dump(summary, f, indent=2)

    save_daily_stock_returns(ret_wide, OUTPUT_DIR)
    save_portfolio_weights(portfolio_df, OUTPUT_DIR)
    save_pnl_paths(pnl, OUTPUT_DIR)

    print('Done. Outputs saved to', OUTPUT_DIR.resolve())
    return summary

print('Runner ready.')

Runner ready.


In [ ]:
def save_daily_stock_returns(ret_wide: pd.DataFrame, out_dir: Path):
    """Save daily returns of all stocks."""
    out_path = out_dir / "stocks_daily_returns.csv"
    ret_wide.to_csv(out_path)
    print(f"[Extra Output] Saved daily stock returns → {out_path}")


def save_portfolio_weights(portfolio_df: pd.DataFrame, out_dir: Path):
    """Save portfolio tickers + weights."""
    out_path = out_dir / "current_portfolio_weights.csv"
    portfolio_df.to_csv(out_path, index=False)
    print(f"[Extra Output] Saved portfolio weights → {out_path}")


def save_pnl_paths(pnl: np.ndarray, out_dir: Path):
    """Save MC scenario P&L paths."""
    out_path = out_dir / "scenario_pnl_paths.csv"
    df = pd.DataFrame({"pnl": pnl})
    df.to_csv(out_path, index=False)
    print(f"[Extra Output] Saved scenario P&L paths → {out_path}")

In [ ]:
SCENARIO_JSON_PATH = "/content/drive/MyDrive/4511 B16/final_state.json"
with open(SCENARIO_JSON_PATH, "r") as f:
    scenario_json = json.load(f)

In [ ]:
summary = run_pipeline(

    scenario_json=scenario_json
)

Mode: EVENT_ONLY; Portfolio size: 50; Scenario keys: ['scenario_text', 'macro_analysis', 'critique', 'revision', 'evaluation']
Portfolio variance (factor, idio, total)= 8.206186104194575e-07 1.1555403339177267e-05 1.2376021949596725e-05
VaR95=0.0047, CVaR95=0.0062
[Extra Output] Saved daily stock returns → /content/drive/MyDrive/4511 B16/Stage2_Outputs/stocks_daily_returns.csv
[Extra Output] Saved portfolio weights → /content/drive/MyDrive/4511 B16/Stage2_Outputs/current_portfolio_weights.csv
[Extra Output] Saved scenario P&L paths → /content/drive/MyDrive/4511 B16/Stage2_Outputs/scenario_pnl_paths.csv
Done. Outputs saved to /content/drive/MyDrive/4511 B16/Stage2_Outputs


# B16 Strategy Designer + Explainer

In [ ]:
# =========================================
# Full Investment Pipeline: Profiling → Strategy → Explainer
# =========================================
# =========================================
# 1️⃣ User Profiler Agent
# =========================================
class UserProfilerAgent:
    def __init__(self, api_key=None):
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        self.api_url = "https://api.groq.com/openai/v1/chat/completions"
        if not self.api_key:
            print("⚠️ Warning: No GROQ_API_KEY found. Free-text inference will use default heuristics.")

    def ask_user(self):
        print("\n💬 Investment Style Profiler")
        print("------------------------------------------------------------------")
        print("Please answer this question to personalize your optimization:")
        print("» What best describes your investment style?")
        print("   a) Very aggressive – I can handle large swings for higher returns")
        print("   b) Moderate – I prefer a balance of risk and return")
        print("   c) Conservative – I value stability and capital preservation")
        print("   Or type your own description (e.g., 'I buy Apple stocks for years')")
        print("------------------------------------------------------------------")
        answer = input("Your choice (a / b / c or free text): ").strip()
        return answer

    def _call_llm_api(self, user_text):
        if not self.api_key:
            return None
        prompt = f"""
You are a financial advisor AI.
User input: "{user_text}"
Please output a JSON with numeric risk profile:
{{
    "risk_appetite": float between 0 and 1,
    "lambda_risk_aversion": float between 0.1 and 5,
    "liquidity_pref": float between 0 and 1,
    "return_focus": float between 0 and 1
}}
Ensure valid JSON only.
"""
        payload = {
            "messages": [{"role": "user", "content": prompt}],
            "model": "llama-3.3-70b-versatile",
            "temperature": 0.2,
            "max_tokens": 200
        }
        headers = {"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"}
        try:
            response = requests.post(self.api_url, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            data = response.json()
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            print(f"⚠️ LLM API call failed: {e}")
            return None

    def infer_lambda(self, user_answer):
        text = user_answer.lower()
        if text.startswith("a") or "aggr" in text:
            return {"risk_appetite": 0.9, "lambda_risk_aversion": 0.5, "liquidity_pref":0.5, "return_focus":0.5}
        elif text.startswith("c") or "cons" in text:
            return {"risk_appetite": 0.3, "lambda_risk_aversion": 4.0, "liquidity_pref":0.5, "return_focus":0.5}
        elif text.startswith("b") or "mod" in text:
            return {"risk_appetite": 0.6, "lambda_risk_aversion": 2.0, "liquidity_pref":0.5, "return_focus":0.5}

        llm_result = self._call_llm_api(user_answer)
        if llm_result:
            try:
                import re
                match = re.search(r"\{.*\}", llm_result, re.DOTALL)
                if match:
                    profile = json.loads(match.group())
                    return {
                        "risk_appetite": float(profile.get("risk_appetite", 0.6)),
                        "lambda_risk_aversion": float(profile.get("lambda_risk_aversion", 2.0)),
                        "liquidity_pref": float(profile.get("liquidity_pref", 0.5)),
                        "return_focus": float(profile.get("return_focus", 0.5))
                    }
            except:
                pass
        return {"risk_appetite": 0.6, "lambda_risk_aversion": 2.0, "liquidity_pref":0.5, "return_focus":0.5}

# =========================================
# 2️⃣ Strategy Designer Agent
# =========================================
class StrategyDesignerAgent:
    def __init__(self, returns_path, summary_path=None, risk_free_rate=0.03, max_assets=30, default_weight=0.02):
        self.returns_path = returns_path
        self.summary_path = summary_path
        self.rf = risk_free_rate
        self.max_assets = max_assets
        self.default_weight = default_weight  # used to detect “empty” portfolio

    def run(self, current_portfolio=None, user_profile=None):
        returns_df = pd.read_csv(self.returns_path, index_col=0).apply(pd.to_numeric, errors="coerce").fillna(0)
        returns_df = returns_df.clip(-0.05, 0.05)  # limit extreme returns

        assets = list(returns_df.columns)
        n_assets = len(assets)

        # --- Stage2 risk metrics ---
        stage2_metrics = {}
        if self.summary_path and os.path.exists(self.summary_path):
            with open(self.summary_path) as f:
                summary = json.load(f)
            stage2_metrics = {
                "portfolio_var_total": summary.get("portfolio_var_total", 0),
                "VaR95": summary.get("VaR95"),
                "CVaR95": summary.get("CVaR95"),
                "stress_scenarios": summary.get("Stress_Scenarios", [])
            }

        if user_profile is None:
            user_profile = {"risk_appetite": 0.6, "lambda_risk_aversion": 2.0, "liquidity_pref":0.5, "return_focus":0.5}

        lam = float(user_profile.get("lambda_risk_aversion", 2.0))
        liquidity_pref = float(user_profile.get("liquidity_pref", 0.5))
        return_focus = float(user_profile.get("return_focus", 0.5))
        alpha = 1.0 + 0.5 * return_focus
        reg_strength = 0.02 * (1 + lam / 5)

        mu_vals = returns_df.mean().values.astype(np.float64)
        Sigma = returns_df.cov()
        Sigma = (Sigma + Sigma.T)/2
        Sigma = Sigma.fillna(0).replace([np.inf, -np.inf], 0)
        liquidity_scale = 1 + (1 - liquidity_pref) * 0.3
        Sigma_scaled = (Sigma * liquidity_scale * 0.9 + np.eye(n_assets)*1e-6).values.astype(np.float64)

        # --- Limit to top max_assets by mean return ---
        top_indices = np.argsort(mu_vals)[-self.max_assets:]
        mu_vals = mu_vals[top_indices]
        Sigma_scaled = Sigma_scaled[np.ix_(top_indices, top_indices)]
        selected_assets = [assets[i] for i in top_indices]

        # --- Convex optimization ---
        w = cp.Variable(len(selected_assets))
        constraints = [cp.sum(w)==1, w>=0, w<=0.25]
        obj = alpha*(mu_vals @ w) - lam/2*cp.quad_form(w, Sigma_scaled) - reg_strength*cp.norm2(w)
        prob = cp.Problem(cp.Maximize(obj), constraints)

        solver_used, success = None, False
        for solver_choice in ["ECOS","SCS","CLARABEL"]:
            try:
                prob.solve(solver=solver_choice)
                if prob.status in ("optimal","optimal_inaccurate"):
                    solver_used = solver_choice
                    success = True
                    break
            except:
                continue

        if not success or w.value is None:
            weights = {a:1/len(selected_assets) for a in selected_assets}
            expected_ret = volatility = sharpe = 0.0
        else:
            w_val = np.clip(w.value,0,1)
            w_val /= w_val.sum()+1e-12
            weights = dict(zip(selected_assets,w_val.round(6)))
            expected_ret = float(mu_vals @ w_val)
            volatility = float(np.sqrt(w_val.T @ Sigma_scaled @ w_val))
            sharpe = (expected_ret - self.rf/252)/(volatility+1e-9)

        # --- Detect empty/default portfolio ---
        if current_portfolio:
            is_empty = all(abs(current_portfolio.get(a,0)-self.default_weight)<1e-6 for a in current_portfolio)
        else:
            is_empty = True

        scenario = "new_portfolio" if is_empty else "existing_portfolio"

        # --- Recommended actions ---
        recommended_actions = {}
        if scenario=="existing_portfolio" and current_portfolio:
            for a in weights:
                prev_w, new_w = current_portfolio.get(a,0), weights.get(a,0)
                delta = new_w-prev_w
                if abs(delta)>1e-4:
                    recommended_actions[a] = f"{'Increase' if delta>0 else 'Reduce'} by {abs(delta)*100:.2f}%"
        else:
            recommended_actions = {a:f"Allocate {w*100:.2f}%" for a,w in weights.items()}

        return {
            "user_profile": user_profile,
            "weights": weights,
            "expected_return": expected_ret,
            "volatility": volatility,
            "sharpe_daily": sharpe,
            "lambda_used": lam,
            "stage2_metrics": stage2_metrics,
            "recommended_actions": recommended_actions,
            "scenario": scenario,
            "solver_used": solver_used,
        }

# =========================================
# 3️⃣ Explainer Agent
# =========================================
class ExplainerAgent:
    def generate_report(self, strategy_output: dict, current_portfolio_path: str = None) -> str:
        lines = ["# 📊 Portfolio Storytelling Report\n"]
        daily_ret = strategy_output.get('expected_return', 0)
        daily_vol = strategy_output.get('volatility', 0)
        lam = strategy_output.get('lambda_used', 0)
        trading_days = 252
        annual_ret = daily_ret * trading_days
        annual_vol = daily_vol * np.sqrt(trading_days)
        sharpe_annual = (annual_ret - 0.03) / (annual_vol+1e-9)
        weights = strategy_output.get("weights", {})
        recommended_actions = strategy_output.get("recommended_actions", {})

        lines += [
            "### 💡 Key Portfolio Metrics",
            "| Metric | Daily | Annualized | Insight |",
            "|:--|--:|--:|:--|",
            f"| Expected Return | {daily_ret*100:.2f}% | {annual_ret*100:.2f}% | Projected daily vs year |",
            f"| Volatility | {daily_vol*100:.2f}% | {annual_vol*100:.2f}% | Risk measure |",
            f"| Sharpe Ratio | {daily_ret/daily_vol:.2f} | {sharpe_annual:.2f} | Risk-adjusted performance |",
            f"| Lambda | {lam:.2f} | - | Higher = more conservative |"
        ]
        lines.append("")

        # --- Include user's current portfolio if provided ---
        current_portfolio = {}
        if current_portfolio_path and os.path.exists(current_portfolio_path):
            df_curr = pd.read_csv(current_portfolio_path, index_col=0)
            if "Weight" in df_curr.columns:
                current_portfolio = df_curr["Weight"].to_dict()

        lines += ["### 🧭 Asset Allocation & Recommended Trades", "Your portfolio distribution:"]
        lines.append("| Asset | Weight | Action vs Current |")
        lines.append("|:--|--:|:--|")
        for asset in sorted(weights, key=lambda x: weights[x], reverse=True):
            w = weights[asset]
            if current_portfolio and strategy_output.get("scenario")=="existing_portfolio":
                curr_w = current_portfolio.get(asset, 0)
                delta = w - curr_w
                if abs(delta)>1e-4:
                    action = f"{'Increase' if delta>0 else 'Reduce'} by {abs(delta)*100:.2f}%"
                else:
                    action = "Hold"
            else:
                action = recommended_actions.get(asset, f"Allocate {w*100:.2f}%")
            symbol = "🟢" if "Increase" in action else "🔻" if "Reduce" in action else "⚙️"
            lines.append(f"| {asset} | {w*100:.2f}% | {symbol} {action} |")

        risk = strategy_output.get("stage2_metrics", {})
        if risk:
            lines += ["\n### ⚠️ Stage2 Risk Insights", "Additional risk indicators:"]
            lines.append("| Metric | Value | Interpretation |")
            lines.append("|:--|--:|:--|")
            for k,v in risk.items():
                if isinstance(v,list): v = ", ".join(map(str,v))
                lines.append(f"| {k} | {v} | Monitor extreme scenarios |")

        # --- Explicit tip for empty/default portfolio ---
        if strategy_output.get("scenario")=="new_portfolio" and current_portfolio:
            lines.append("\n⚠️ You currently hold only default allocations; treat this as a fresh portfolio and follow the recommended allocations.\n")

        lines += [
            "\n### ♥️Our investment solution only contains 30 stocks in maximum for the simplicity of real-life operation.♥️"
            "\n### 📢 Bottom Line",
            f"This {strategy_output.get('scenario','new_portfolio').replace('_',' ')} is based on your risk profile. 📚",
            "Follow recommended allocations, review periodically, and adjust as needed. 💡📈🧭",
            "⚠️ Overfitting warning: historical returns may exaggerate projected gains.",
            "💡 Tip: Think of your portfolio as a living tutorial for risk & reward!"
        ]
        return "\n".join(lines)


In [ ]:
API_KEY = "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"
returns_file = "/content/drive/MyDrive/4511 B16/Stage2_Outputs/stocks_daily_returns.csv"
summary_file = "/content/drive/MyDrive/4511 B16/Stage2_Outputs/summary.json"
current_weights_file = "/content/drive/MyDrive/4511 B16/Stage2_Outputs/current_portfolio_weights.csv"

profiler = UserProfilerAgent(api_key=API_KEY)
answer = profiler.ask_user()
user_profile = profiler.infer_lambda(answer)
print("\n✅ Inferred User Profile:", user_profile)

# --- Load current portfolio weights ---
current_portfolio = {}
if os.path.exists(current_weights_file):
    df_curr = pd.read_csv(current_weights_file, index_col=0)
    if "Weight" in df_curr.columns:
        current_portfolio = df_curr["Weight"].to_dict()

strategy_agent = StrategyDesignerAgent(returns_path=returns_file, summary_path=summary_file)
strategy_output = strategy_agent.run(current_portfolio=current_portfolio, user_profile=user_profile)

explainer = ExplainerAgent()
report = explainer.generate_report(strategy_output, current_portfolio_path=current_weights_file)
print("\n" + report)


💬 Investment Style Profiler
------------------------------------------------------------------
Please answer this question to personalize your optimization:
» What best describes your investment style?
   a) Very aggressive – I can handle large swings for higher returns
   b) Moderate – I prefer a balance of risk and return
   c) Conservative – I value stability and capital preservation
   Or type your own description (e.g., 'I buy Apple stocks for years')
------------------------------------------------------------------
Your choice (a / b / c or free text): b

✅ Inferred User Profile: {'risk_appetite': 0.6, 'lambda_risk_aversion': 2.0, 'liquidity_pref': 0.5, 'return_focus': 0.5}

# 📊 Portfolio Storytelling Report

### 💡 Key Portfolio Metrics
| Metric | Daily | Annualized | Insight |
|:--|--:|--:|:--|
| Expected Return | 0.09% | 23.63% | Projected daily vs year |
| Volatility | 1.10% | 17.54% | Risk measure |
| Sharpe Ratio | 0.08 | 1.18 | Risk-adjusted performance |
| Lambda | 2.00 